# Prompt Optimization for Multi-Model Compatibility

This notebook teaches you how to write prompts that work consistently across different LLM providers. You'll learn:

1. Why the same prompt produces different outputs on different models
2. How to profile model "personalities" and default behaviors
3. Techniques for writing portable, efficient prompts
4. **Special handling for reasoning models** (chain-of-thought models like GPT-OSS)

![Prompt Drift and Prediction Drift](data/prompt-optimers.png)

## Models Tested

| Model | ID | Type |
|-------|-----|------|
| Amazon Nova 2 Lite | `us.amazon.nova-2-lite-v1:0` | AWS Native |
| Claude 3.5 Haiku | `us.anthropic.claude-3-5-haiku-20241022-v1:0` | Anthropic |
| GPT-OSS 20B | `openai.gpt-oss-20b-1:0` | Reasoning Model |
| Qwen3 Coder | `qwen.qwen3-coder-30b-a3b-v1:0` | Alibaba |

## Setup

In [ ]:
import boto3
import json
import time
from typing import Dict, List, Optional

# Initialize Bedrock client
REGION = "us-east-1"
bedrock = boto3.client('bedrock-runtime', region_name=REGION)

# Define our test models
MODELS = {
    "Nova 2 Lite": "us.amazon.nova-2-lite-v1:0",
    "Claude Haiku": "us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "GPT-OSS 20B": "openai.gpt-oss-20b-1:0",
    "Qwen3 Coder": "qwen.qwen3-coder-30b-a3b-v1:0",
}

def invoke_model(model_id: str, prompt: str, system: str = "You are a helpful assistant.", 
                 max_tokens: int = 300) -> Dict:
    """
    Invoke a model and return structured results.
    Handles both standard and reasoning models (with chain-of-thought).
    """
    try:
        start = time.time()
        
        response = bedrock.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            system=[{"text": system}],
            inferenceConfig={"maxTokens": max_tokens, "temperature": 0.1}
        )
        
        latency = (time.time() - start) * 1000
        
        # Parse content - handle reasoning models with chain-of-thought
        content_blocks = response['output']['message']['content']
        output_text = ""
        reasoning_text = ""
        
        for block in content_blocks:
            if 'text' in block:
                output_text = block['text']
            elif 'reasoningContent' in block:
                reasoning_text = block['reasoningContent'].get('reasoningText', {}).get('text', '')
        
        usage = response.get('usage', {})
        
        return {
            "success": True,
            "output": output_text,
            "reasoning": reasoning_text,
            "input_tokens": usage.get('inputTokens', 0),
            "output_tokens": usage.get('outputTokens', 0),
            "latency_ms": latency,
            "stop_reason": response.get('stopReason', 'unknown'),
            "has_reasoning": bool(reasoning_text)
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

print("✅ Setup complete")
print(f"   Region: {REGION}")
print(f"   Models: {len(MODELS)}")

---

## Part 1: The Problem with Prompts

Let's see what happens when we send the **exact same prompt** to four different models.

### The Prompt

We'll use a simple healthcare prompt that asks for brevity:

> "Summarize the side effects of metformin briefly."

The word "briefly" is subjective. Each model will interpret it differently.

In [ ]:
# Test the same prompt across all models
VAGUE_PROMPT = "Summarize the side effects of metformin briefly."

print("PROMPT:", VAGUE_PROMPT)
print("=" * 80)

vague_results = {}

for name, model_id in MODELS.items():
    result = invoke_model(model_id, VAGUE_PROMPT)
    vague_results[name] = result
    
    if result["success"]:
        print(f"\n{'─' * 80}")
        print(f"🤖 {name} ({result['output_tokens']} tokens, {result['latency_ms']:.0f}ms)")
        print('─' * 80)
        
        if result.get("has_reasoning"):
            print(f"💭 [Has hidden reasoning - {len(result['reasoning'])} chars]")
        
        print(result["output"][:500])
        if len(result["output"]) > 500:
            print("...")
    else:
        print(f"\n❌ {name}: {result['error']}")

In [ ]:
# Analyze the differences
print("ANALYSIS: What does 'briefly' mean to each model?")
print("=" * 80)
print(f"\n{'Model':<15} | {'Tokens':>8} | {'Words':>8} | {'Lines':>6} | {'Interpretation'}")
print("-" * 80)

for name, result in vague_results.items():
    if result["success"]:
        output = result["output"]
        tokens = result["output_tokens"]
        words = len(output.split())
        lines = len([l for l in output.split('\n') if l.strip()])
        
        # Characterize the response
        if tokens > 150:
            interp = "Comprehensive explanation"
        elif tokens > 80:
            interp = "Moderate summary"
        else:
            interp = "Concise list"
            
        print(f"{name:<15} | {tokens:>8} | {words:>8} | {lines:>6} | {interp}")

# Calculate variance dynamically
tokens_list = [r["output_tokens"] for r in vague_results.values() if r["success"]]
min_tokens, max_tokens = min(tokens_list), max(tokens_list)
variance_ratio = max_tokens / min_tokens

print("\n" + "=" * 80)
print(f"TOKEN VARIANCE: {min_tokens} to {max_tokens} ({variance_ratio:.1f}x difference!)")
print(f"\nKEY INSIGHT: 'Briefly' is interpreted as anywhere from {min_tokens} to {max_tokens} tokens.")
print("             This is why prompts are not portable across models.")

---

## Part 2: Model Personality Profiles

Each model has a distinct "personality" - default behaviors for verbosity, formatting, and interpretation. Understanding these helps you write better prompts.

In [ ]:
# Profile each model's formatting tendencies
print("MODEL PERSONALITY PROFILES")
print("=" * 80)

for name, result in vague_results.items():
    if not result["success"]:
        continue
        
    output = result["output"]
    
    # Analyze formatting
    has_headers = "##" in output or "###" in output
    has_bold = "**" in output
    has_bullets = "- " in output or "• " in output
    has_numbers = any(f"{i}." in output for i in range(1, 10))
    has_tables = "|" in output and "---" in output
    has_tips = "tip" in output.lower() or "note" in output.lower()
    has_reasoning = result.get("has_reasoning", False)
    
    print(f"\n{'─' * 60}")
    print(f"🤖 {name}")
    print('─' * 60)
    print(f"   Tokens: {result['output_tokens']}")
    print(f"   Latency: {result['latency_ms']:.0f}ms")
    print(f"   ")
    print(f"   Formatting:")
    print(f"     • Headers (##):      {'✅' if has_headers else '❌'}")
    print(f"     • Bold (**):         {'✅' if has_bold else '❌'}")
    print(f"     • Bullet points:     {'✅' if has_bullets else '❌'}")
    print(f"     • Numbered lists:    {'✅' if has_numbers else '❌'}")
    print(f"     • Tables:            {'✅' if has_tables else '❌'}")
    print(f"     • Tips/Notes:        {'✅' if has_tips else '❌'}")
    print(f"     • Chain-of-Thought:  {'✅ REASONING MODEL' if has_reasoning else '❌'}")

In [ ]:
# Generate dynamic summary table based on actual results
print("\nMODEL PERSONALITY SUMMARY")
print("=" * 80)

# Build dynamic table
print(f"\n{'Model':<15} | {'Verbosity':<18} | {'Key Formats':<20} | {'Special':<20}")
print("-" * 80)

for name, result in vague_results.items():
    if not result["success"]:
        continue
    
    output = result["output"]
    tokens = result["output_tokens"]
    
    # Determine verbosity level
    if tokens > 200:
        verbosity = f"HIGH ({tokens} tokens)"
    elif tokens > 120:
        verbosity = f"MEDIUM ({tokens} tokens)"
    else:
        verbosity = f"LOW ({tokens} tokens)"
    
    # Detect formats used
    formats = []
    if "##" in output or "###" in output:
        formats.append("headers")
    if "**" in output:
        formats.append("bold")
    if "- " in output or "• " in output:
        formats.append("bullets")
    if "|" in output and "---" in output:
        formats.append("tables")
    format_str = ", ".join(formats) if formats else "plain text"
    
    # Special behavior
    if result.get("has_reasoning"):
        special = "⚠️ REASONING MODEL"
    elif "tip" in output.lower() or "note" in output.lower():
        special = "Adds tips/notes"
    elif tokens > 200:
        special = "Very verbose"
    elif tokens < 100:
        special = "Naturally concise"
    else:
        special = "Balanced"
    
    print(f"{name:<15} | {verbosity:<18} | {format_str:<20} | {special:<20}")

print("\n" + "=" * 80)
print("⚠️  Note: GPT-OSS 20B is a REASONING model - it thinks before responding.")
print("         This affects how it responds to format constraints.")

---

## Part 3: Writing Portable Prompts

The key to portable prompts is **explicit constraints**. Don't say "briefly" - say exactly what you want.

### Three Constraint Techniques

1. **Explicit limits**: "Maximum 3 sentences" instead of "briefly"
2. **Structured output**: Specify the exact format you want
3. **Negative examples**: Tell the model what NOT to do

In [ ]:
# Test explicit constraints
EXPLICIT_PROMPT = "List the 5 most common metformin side effects. One line each. No explanations."

print("TECHNIQUE 1: EXPLICIT LIMITS")
print("=" * 80)
print(f"Prompt: \"{EXPLICIT_PROMPT}\"")
print()

explicit_results = {}

for name, model_id in MODELS.items():
    result = invoke_model(model_id, EXPLICIT_PROMPT)
    explicit_results[name] = result
    
    if result["success"]:
        print(f"\n{name} ({result['output_tokens']} tokens):")
        print(result["output"])

In [ ]:
# Test structured output
STRUCTURED_PROMPT = """List metformin side effects in this exact format:
COMMON: [3 items, comma-separated]
SERIOUS: [2 items, comma-separated]
Nothing else."""

print("TECHNIQUE 2: STRUCTURED OUTPUT")
print("=" * 80)
print(f"Prompt:\n{STRUCTURED_PROMPT}")
print()

structured_results = {}

for name, model_id in MODELS.items():
    result = invoke_model(model_id, STRUCTURED_PROMPT)
    structured_results[name] = result
    
    if result["success"]:
        print(f"\n{name} ({result['output_tokens']} tokens):")
        print(result["output"])

In [ ]:
# Compare token reduction
print("\nTOKEN REDUCTION COMPARISON")
print("=" * 80)
print(f"\n{'Model':<15} | {'Vague':>8} | {'Explicit':>8} | {'Structured':>10} | {'Reduction':>10}")
print("-" * 65)

reductions = []
gpt_reduction = None

for name in MODELS.keys():
    vague = vague_results.get(name, {}).get("output_tokens", 0)
    explicit = explicit_results.get(name, {}).get("output_tokens", 0)
    structured = structured_results.get(name, {}).get("output_tokens", 0)

    if vague > 0 and structured > 0:
        reduction = (1 - structured / vague) * 100
        status = f"{reduction:.0f}%" if reduction > 0 else f"{reduction:.0f}% ⚠️"
        print(f"{name:<15} | {vague:>8} | {explicit:>8} | {structured:>10} | {status:>10}")

        # Track reductions
        if "GPT" in name:
            gpt_reduction = reduction
        elif reduction > 0:
            reductions.append(reduction)

# Dynamic insight based on actual results
if reductions:
    min_red, max_red = min(reductions), max(reductions)
    print("\n" + "=" * 80)
    print(f"KEY INSIGHT: Explicit constraints reduce tokens by {min_red:.0f}-{max_red:.0f}% for standard models.")
    if gpt_reduction is not None and gpt_reduction < min_red:
        print(f"             But notice GPT-OSS 20B ({gpt_reduction:.0f}%) - reasoning models respond differently!")

---

## Part 4: The Reasoning Model Challenge - Contraindication Screening

### When to Use Reasoning Models

Reasoning models like GPT-OSS 20B have **chain-of-thought** capabilities - they "think" before responding. This is powerful for complex tasks but comes with tradeoffs:

| Aspect | Standard Models | Reasoning Models |
|--------|-----------------|------------------|
| Processing | Direct response | Think first, then reply |
| Thinking tokens | None | Hidden reasoning tokens (costs $) |
| Format constraints | Work well | Less effective |
| Best for | Simple, formatted tasks | Complex reasoning |

### The Optimization Opportunity

**Key insight**: You can reduce thinking tokens by embedding explicit rules, or provide explicit assumptions in your prompt.

- **Vague prompt**: Model must DERIVE rules from training (more thinking)
- **Optimized prompt**: Model just CHECKS rules you provide (less thinking)

We'll demonstrate this with a **medical contraindication screening** use case.

In [ ]:
# Load contraindication screening data
import json
import re

# Helper function to extract JSON (define early for use in this section)
def extract_json(text):
    """Extract JSON from text, handling markdown blocks."""
    text = text.strip()
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        text = match.group(1).strip()
    return json.loads(text)

# Load patient profile
with open('data/contraindication_patient.txt', 'r') as f:
    PATIENT_PROFILE = f.read()

# Load expected output (golden response)
with open('data/contraindication_expected_output.json', 'r') as f:
    CONTRAINDICATION_EXPECTED = json.load(f)

# Load prompts
import sys
sys.path.insert(0, 'data')
from medical_prompts import get_simple_contraindication_prompt, get_optimized_contraindication_prompt

print("CONTRAINDICATION SCREENING TEST")
print("=" * 80)
print("\nPatient has multiple high-risk conditions. We're screening 5 proposed medications.")

print("\n" + "-" * 80)
print("PATIENT PROFILE:")
print("-" * 80)
print(PATIENT_PROFILE)

print("\n" + "-" * 80)
print("EXPECTED RESULTS:")
print("-" * 80)
print(f"  Contraindicated: {CONTRAINDICATION_EXPECTED['summary']['contraindicated']} medications")
print(f"  Caution: {CONTRAINDICATION_EXPECTED['summary']['caution']} medications")
print(f"  Safe: {CONTRAINDICATION_EXPECTED['summary']['safe']} medications")

In [ ]:
# Test GPT-OSS with simple (vague) prompt vs optimized prompt
GPT_MODEL_ID = "openai.gpt-oss-20b-1:0"

def validate_contraindication_text(output: str, expected: dict) -> dict:
    """Validate contraindication results from text output."""
    results = {"score": 0, "max": 5, "errors": []}
    output_upper = output.upper()

    # Check each expected drug
    for exp in expected.get("results", []):
        drug = exp["drug"].upper().split()[0]  # First word of drug name
        status = exp["status"].upper()

        # Check if drug is mentioned with correct status
        if drug in output_upper:
            if status in output_upper:
                results["score"] += 1
            else:
                results["errors"].append(f"{drug}: status not found")
        else:
            results["errors"].append(f"Missing: {drug}")

    return results

print("STEP 1: Testing with SIMPLE (Vague) Prompt")
print("=" * 80)
print("Prompt style: 'Check if medications are safe' - model must derive rules")

simple_prompt = get_simple_contraindication_prompt(PATIENT_PROFILE)

print(f"\nPrompt length: {len(simple_prompt)} characters")
print("\n" + "-" * 40)

simple_result = invoke_model(GPT_MODEL_ID, simple_prompt, max_tokens=3000)

simple_score = 0
simple_thinking_len = 0

if simple_result["success"]:
    simple_thinking_len = len(simple_result.get("reasoning", ""))
    print(f"  Output tokens: {simple_result['output_tokens']}")
    print(f"  Thinking chars: {simple_thinking_len}")
    print(f"  Latency: {simple_result['latency_ms']:.0f}ms")

    if simple_result.get("reasoning"):
        print(f"\n  Thinking preview:")
        print(f"  \"{simple_result['reasoning'][:200]}...\"")

    # Validate output
    validation = validate_contraindication_text(simple_result["output"], CONTRAINDICATION_EXPECTED)
    simple_score = validation['score'] / validation['max'] * 100 if validation['max'] > 0 else 0
    status = "PASS" if simple_score >= 80 else "FAIL"
    print(f"\n  Accuracy: {validation['score']}/{validation['max']} ({simple_score:.0f}%) {status}")
    if validation['errors']:
        for err in validation['errors'][:3]:
            print(f"    - {err}")

    print(f"\n  Output preview:")
    print(f"  \"{simple_result['output'][:300]}...\"")
else:
    print(f"  Error: {simple_result.get('error')}")

In [ ]:
# Test with OPTIMIZED prompt (explicit rules, simple output)
print("\nSTEP 2: Testing with OPTIMIZED Prompt (Explicit Rules)")
print("=" * 80)
print("Prompt style: Explicit rules + simple output format - model just checks patterns")

optimized_prompt = get_optimized_contraindication_prompt(PATIENT_PROFILE)

print(f"\nPrompt length: {len(optimized_prompt)} characters")
print("\n" + "-" * 40)

# Use same max_tokens as simple prompt to ensure fair comparison
optimized_result = invoke_model(GPT_MODEL_ID, optimized_prompt, max_tokens=1500)

optimized_score = 0
optimized_thinking_len = 0

if optimized_result["success"]:
    optimized_thinking_len = len(optimized_result.get("reasoning", ""))
    print(f"  Output tokens: {optimized_result['output_tokens']}")
    print(f"  Thinking chars: {optimized_thinking_len}")
    print(f"  Latency: {optimized_result['latency_ms']:.0f}ms")

    if optimized_result.get("reasoning"):
        print(f"\n  Thinking preview:")
        print(f"  \"{optimized_result['reasoning'][:200]}...\"")

    # Validate output
    validation = validate_contraindication_text(optimized_result["output"], CONTRAINDICATION_EXPECTED)
    optimized_score = validation['score'] / validation['max'] * 100 if validation['max'] > 0 else 0
    status = "PASS" if optimized_score >= 80 else "FAIL"
    print(f"\n  Accuracy: {validation['score']}/{validation['max']} ({optimized_score:.0f}%) {status}")
    if validation['errors']:
        for err in validation['errors'][:3]:
            print(f"    - {err}")

    print(f"\n  Output preview:")
    print(f"  \"{optimized_result['output'][:300]}...\"")
else:
    print(f"  Error: {optimized_result.get('error')}")

# Calculate improvement
print("\n" + "=" * 80)
print("COMPARISON: Simple vs Optimized Prompt")
print("=" * 80)

thinking_reduction = 0
if simple_thinking_len > 0:
    thinking_reduction = (1 - optimized_thinking_len / simple_thinking_len) * 100

output_reduction = 0
if simple_result.get('output_tokens', 0) > 0:
    output_reduction = (1 - optimized_result.get('output_tokens', 0) / simple_result.get('output_tokens', 0)) * 100

print(f"\n{'Metric':<25} | {'Simple':>15} | {'Optimized':>15} | {'Reduction':>15}")
print("-" * 75)
print(f"{'Thinking (chars)':<25} | {simple_thinking_len:>15,} | {optimized_thinking_len:>15,} | {thinking_reduction:>+14.0f}%")
print(f"{'Output tokens':<25} | {simple_result.get('output_tokens', 0):>15} | {optimized_result.get('output_tokens', 0):>15} | {output_reduction:>+14.0f}%")
print(f"{'Accuracy':<25} | {simple_score:>14.0f}% | {optimized_score:>14.0f}% | {'':>15}")

if thinking_reduction > 0:
    print(f"\n✅ Optimized prompt reduced thinking by {thinking_reduction:.0f}%!")
else:
    print(f"\n⚠️ Thinking increased by {-thinking_reduction:.0f}% - consider simplifying the rules.")

### Key Takeaways: Optimizing Reasoning Models

| Optimization Strategy | Description | Impact |
|-----------------------|-------------|--------|
| **Embed explicit rules** | Provide contraindication rules directly in prompt | Reduces thinking by 30-50% |
| **Pattern matching > Derivation** | Let model CHECK rules, not DERIVE them | Faster, more accurate |
| **Structured output format** | Provide exact JSON template | Consistent output |
| **Rule-based prompts** | Works like code review (match patterns) | Better than decision trees |

#### When to Use Reasoning Models

| Use Case | Reasoning Model? | Why |
|----------|-----------------|-----|
| Drug interaction checking | Yes | Multiple rules to apply |
| Simple data extraction | No | Overkill, use standard model |
| Complex medical analysis | Yes | Needs chain-of-thought |
| Format conversion | No | Standard models are cheaper |

> **Cost Tip**: Thinking tokens are hidden but still cost money. Embedding rules in prompts reduces thinking tokens by 30-50%, directly reducing costs while maintaining or improving accuracy.

---

## Part 5: Complex Medical Data Extraction

This section demonstrates prompt optimization with a **real-world medical extraction challenge**:

- A **massive, noisy medical report** with 7+ physicians, 20+ dates, and multiple ID numbers
- The model must identify the **correct attending physician** among many referenced providers
- Extract **only medications ordered for this admission** (not home medications)
- Navigate through equipment notes, facility metadata, and historical records

### The Challenge

| Difficulty Factor | Description |
|------------------|-------------|
| Multiple physicians | 7+ doctors with NPIs - must identify the ATTENDING |
| Multiple dates | 20+ dates - must find the VISIT date |
| Multiple IDs | MRN, Insurance ID, Encounter ID, etc. |
| Medication noise | Home meds, held meds, and admission meds mixed |
| Nested structure | 4-level deep JSON with arrays |

In [ ]:
# Load the medical extraction data from separate files
import sys
import json

sys.path.insert(0, 'data')

# Load medical report
with open('data/medical_report.txt', 'r') as f:
    MEDICAL_REPORT = f.read()

# Load expected output
with open('data/medical_expected_output.json', 'r') as f:
    MEDICAL_EXPECTED = json.load(f)

# Load prompts
from medical_prompts import get_natural_language_prompt

# Validation function to compare output against expected
def validate_extraction(output: dict, expected: dict, path: str = "") -> dict:
    """Deep compare output against expected, tracking matches."""
    results = {"score": 0, "max": 0, "errors": []}

    def compare(out, exp, p):
        if isinstance(exp, dict):
            for key, val in exp.items():
                new_path = f"{p}.{key}" if p else key
                if key not in out:
                    results["errors"].append(f"Missing: {new_path}")
                    results["max"] += 1
                else:
                    compare(out[key], val, new_path)
        elif isinstance(exp, list):
            results["max"] += 1
            if not isinstance(out, list):
                results["errors"].append(f"Expected list at {p}")
            elif len(out) < len(exp):
                results["errors"].append(f"Expected {len(exp)} items at {p}, got {len(out)}")
            else:
                results["score"] += 1
                for i, item in enumerate(exp):
                    if i < len(out):
                        compare(out[i], item, f"{p}[{i}]")
        else:
            results["max"] += 1
            out_val = str(out).lower().strip() if out else ""
            exp_val = str(exp).lower().strip()
            if out_val == exp_val or exp_val in out_val:
                results["score"] += 1
            else:
                results["errors"].append(f"Mismatch at {p}: expected '{exp}', got '{out}'")

    compare(output, expected, path)
    return results

print("MEDICAL DATA EXTRACTION TEST")
print("=" * 80)
print(f"\nMedical Report: {len(MEDICAL_REPORT):,} characters")
print(f"Expected Output: {len(MEDICAL_EXPECTED)} top-level fields")

# Show truncated preview
print("\n" + "─" * 80)
print("MEDICAL REPORT (truncated):")
print("─" * 80)
print(MEDICAL_REPORT[:1200])
print("\n... [truncated - full report is {:,} chars] ...".format(len(MEDICAL_REPORT)))

print("\n" + "─" * 80)
print("EXPECTED OUTPUT STRUCTURE:")
print("─" * 80)
print(json.dumps(MEDICAL_EXPECTED, indent=2))


In [ ]:
# Test with Claude Opus 4.5 and Nova 2 Lite using natural language prompt
OPUS_MODEL_ID = "us.anthropic.claude-opus-4-5-20251101-v1:0"
NOVA_MODEL_ID = "us.amazon.nova-2-lite-v1:0"

natural_prompt = get_natural_language_prompt(MEDICAL_REPORT)

print("STEP 1: Testing Natural Language Prompt")
print("=" * 80)
print("\nPrompt style: Describes what to extract in prose, no explicit field names")

def extract_json(text):
    """Extract JSON from text, handling markdown blocks."""
    import re
    text = text.strip()
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        text = match.group(1).strip()
    return json.loads(text)

# Test Opus with natural language prompt
print("\n" + "─" * 60)
print("Claude Opus 4.5 (Natural Language Prompt):")
print("─" * 60)

opus_natural = invoke_model(OPUS_MODEL_ID, natural_prompt, max_tokens=1000)
opus_score = 0
if opus_natural["success"]:
    print(f"  Tokens: {opus_natural['output_tokens']} | Latency: {opus_natural['latency_ms']:.0f}ms")
    try:
        opus_json = extract_json(opus_natural["output"])
        validation = validate_extraction(opus_json, MEDICAL_EXPECTED)
        opus_score = validation['score'] / validation['max'] * 100 if validation['max'] > 0 else 0
        status = "✅ PASS" if opus_score >= 80 else "❌ FAIL"
        print(f"  Score: {validation['score']}/{validation['max']} ({opus_score:.0f}%) {status}")
        if validation['errors'][:3]:
            for err in validation['errors'][:3]:
                print(f"    - {err}")
    except Exception as e:
        print(f"  ⚠️ JSON parsing issue: {e}")
else:
    print(f"  ❌ Error: {opus_natural.get('error')}")

# Test Nova with natural language prompt
print("\n" + "─" * 60)
print("Nova 2 Lite (Natural Language Prompt):")
print("─" * 60)

nova_natural = invoke_model(NOVA_MODEL_ID, natural_prompt, max_tokens=1000)
nova_nat_score = 0
if nova_natural["success"]:
    print(f"  Tokens: {nova_natural['output_tokens']} | Latency: {nova_natural['latency_ms']:.0f}ms")
    try:
        nova_json = extract_json(nova_natural["output"])
        validation = validate_extraction(nova_json, MEDICAL_EXPECTED)
        nova_nat_score = validation['score'] / validation['max'] * 100 if validation['max'] > 0 else 0
        status = "✅ PASS" if nova_nat_score >= 80 else "❌ FAIL"
        print(f"  Score: {validation['score']}/{validation['max']} ({nova_nat_score:.0f}%) {status}")
        if validation['errors'][:3]:
            for err in validation['errors'][:3]:
                print(f"    - {err}")
    except Exception as e:
        print(f"  ⚠️ JSON parsing issue: {e}")
else:
    print(f"  ❌ Error: {nova_natural.get('error')}")


In [ ]:
# Test with optimized prompt (markdown + explicit JSON structure)
print("STEP 2: Testing Optimized Prompt (Markdown + JSON Template)")
print("=" * 80)
print("\nPrompt style: Markdown headers, explicit field names, JSON template")

# Test Nova with optimized prompt
print("\n" + "─" * 60)
print("Nova 2 Lite (Optimized Prompt):")
print("─" * 60)
# Load prompts
from medical_prompts import get_optimized_prompt

optimized_prompt = get_optimized_prompt(MEDICAL_REPORT)
optimized_prompt = f"""
## TASK ##
Extract data from the medical report into a JSON object.

## IMPORTANT INSTRUCTIONS ##
- Extract ONLY medications ordered for THIS admission (not home medications)
- The ATTENDING physician is the one responsible for emergency care
- Use the visit/encounter date, not document creation date
- Use the MRN (Medical Record Number), not other IDs
- Encounter type only choices are **emergency** or **outpatient**

## EXACT JSON STRUCTURE ##
```json
{{
  "patient": {{
    "full_legal_name": "full legal name",
    "date_of_birth": "YYYY-MM-DD",
    "age": number,
    "biological_sex": "female" or "male",
    "medical_record_number": "MRN-XXXXXXX"
  }},
  "encounter": {{
    "date": "YYYY-MM-DD",
    "type": "emergency" or "outpatient",
    "hospital": {{
      "name": "hospital name",
      "department": "department name",
      "unit": "unit name"
    }},
    "provider": {{
      "name": "Dr. Full Name",
      "specialty": "specialty",
      "npi": "NPI number"
    }}
  }},
  "diagnosis": {{
    "primary": {{
      "condition": "diagnosis name",
      "icd_10_code": "ICD-10 code",
      "severity": "mild" or "moderate" or "severe"
    }},
    "secondary": [
      {{"condition": "name", "icd_10_code": "code"}}
    ]
  }},
  "treatment": {{
    "medications": [
      {{"name": "drug", "dose": "dose", "route": "oral/iv", "frequency": "frequency"}}
    ],
    "procedures": [
      {{"name": "procedure", "date": "YYYY-MM-DD", "urgency": "urgent/routine"}}
    ],
    "disposition": "disposition in lowercase"
  }}
}}
```

## MEDICAL REPORT ##
{MEDICAL_REPORT}

## RESPONSE ##
Return ONLY the JSON object. No markdown, no explanation.
"""

nova_optimized = invoke_model(NOVA_MODEL_ID, optimized_prompt, max_tokens=1000)
nova_opt_score = 0
if nova_optimized["success"]:
    print(f"  Tokens: {nova_optimized['output_tokens']} | Latency: {nova_optimized['latency_ms']:.0f}ms")
    try:
        nova_opt_json = extract_json(nova_optimized["output"])
        validation = validate_extraction(nova_opt_json, MEDICAL_EXPECTED)
        nova_opt_score = validation['score'] / validation['max'] * 100 if validation['max'] > 0 else 0
        status = "✅ PASS" if nova_opt_score >= 80 else "❌ FAIL"
        print(f"  Score: {validation['score']}/{validation['max']} ({nova_opt_score:.0f}%) {status}")
        if validation['errors'][:3]:
            for err in validation['errors'][:3]:
                print(f"    - {err}")
    except Exception as e:
        print(f"  ⚠️ JSON parsing issue: {e}")
else:
    print(f"  ❌ Error: {nova_optimized.get('error')}")


In [ ]:
# Final comparison with scores
print("MEDICAL EXTRACTION RESULTS SUMMARY")
print("=" * 80)
print()
print(f"{'Model':<20} | {'Prompt':<25} | {'Score':<10} | {'Result':<10}")
print("-" * 70)

# Opus with natural language
opus_status = "✅ PASS" if opus_score >= 80 else "❌ FAIL"
print(f"{'Claude Opus 4.5':<20} | {'Natural Language':<25} | {opus_score:>5.0f}%    | {opus_status:<10}")

# Nova with natural language
nova_nat_status = "✅ PASS" if nova_nat_score >= 80 else "❌ FAIL"
print(f"{'Nova 2 Lite':<20} | {'Natural Language':<25} | {nova_nat_score:>5.0f}%    | {nova_nat_status:<10}")

# Nova with optimized
nova_opt_status = "✅ PASS" if nova_opt_score >= 80 else "❌ FAIL"
print(f"{'Nova 2 Lite':<20} | {'Optimized (Markdown+JSON)':<25} | {nova_opt_score:>5.0f}%    | {nova_opt_status:<10}")

print()
print("=" * 80)
print("KEY INSIGHT:")
print("  - Opus 4.5 handles vague prompts through superior reasoning")
print("  - Nova 2 Lite struggles with natural language (wrong field names, missed context)")
print("  - With markdown formatting + explicit JSON template, Nova 2 Lite matches Opus quality")
print()
print("OPTIMIZATION TECHNIQUES USED:")
print("  1. Markdown headers (## TASK ##, ## INSTRUCTIONS ##)")
print("  2. Explicit field names in backticks")
print("  3. JSON template showing exact structure")
print("  4. Clear disambiguation rules (ATTENDING physician, not ED physician)")


---

## Summary: Prompt Optimization Checklist

### Key Takeaways

1. **"Briefly" is not portable** - Each model interprets vague instructions differently (82-200 tokens)

2. **Explicit constraints work** - For standard models, specific limits reduce tokens by 67-88%

3. **Reasoning models are different** - GPT-OSS 20B has chain-of-thought that affects prompt response

4. **Match model to task** - Don't use reasoning models for simple tasks

### Prompt Optimization Techniques

| Technique | Example | Effectiveness |
|-----------|---------|---------------|
| **Explicit limits** | "Maximum 3 sentences" | High for standard models |
| **Structured output** | "Format: COMMON: [x,y,z]" | Highest for standard models |
| **Negative examples** | "No preamble, no explanations" | Medium |
| **Task matching** | Use Qwen for simple tasks | Cost optimization |

### Reasoning Model Guidelines

| Do | Don't |
|----|-------|
| Use for complex analysis | Use for simple formatting |
| Expect higher token usage | Expect format constraints to work |
| Budget for thinking tokens | Assume same behavior as standard models |

---

**Next**: Proceed to Notebook 4 (Shadow Testing) to learn how to safely migrate between models using parallel testing.